# NLP — Unit 5 Practice Programs
### Chatbot | Question Answering | Summarization | Machine Translation
---
> **Tip:** Runtime → Change runtime type → **GPU** for faster loading.

## Cell 1 — Install Dependencies
> Pinned to `transformers==4.44.2` for full pipeline compatibility.

In [ ]:
!pip install -q transformers==4.44.2 torch sentencepiece
print('All libraries installed successfully!')

## Cell 2 — Imports

In [ ]:
from transformers import pipeline
import warnings, torch
warnings.filterwarnings('ignore')
device = 0 if torch.cuda.is_available() else -1
print('GPU available:', torch.cuda.is_available())
print('Imports done!')

---
# Program 1 — Conversational Chatbot
**Model:** `microsoft/DialoGPT-small`  
Maintains conversation history across turns.

In [ ]:
print('Loading chatbot model...')
chatbot = pipeline('text-generation', model='microsoft/DialoGPT-small', device=device)
print('Chatbot model ready!')

In [ ]:
def chat_once(user_input, history=''):
    prompt = history + 'User: ' + user_input + '\nBot:'
    out = chatbot(prompt, max_new_tokens=80, do_sample=True,
                  temperature=0.75, pad_token_id=50256)
    reply = out[0]['generated_text'].split('Bot:')[-1].strip()
    reply = reply.split('User:')[0].strip()
    return reply

def multi_turn_chat(turns):
    history = ''
    print('=' * 60)
    print('  CHATBOT SESSION')
    print('=' * 60)
    for user_msg in turns:
        reply = chat_once(user_msg, history)
        print('  You :', user_msg)
        print('  Bot :', reply)
        print('-' * 60)
        history += 'User: ' + user_msg + '\nBot: ' + reply + '\n'

conversation = [
    'Hi! What is Natural Language Processing?',
    'Can you give me an example?',
    'What are transformers in NLP?',
    'Thank you! Goodbye.',
]
multi_turn_chat(conversation)

---
# Program 2 — Question Answering
**Model:** `deepset/roberta-base-squad2`  
Extracts exact answer spans from a context passage.

In [ ]:
print('Loading QA model...')
qa = pipeline('question-answering', model='deepset/roberta-base-squad2')
print('QA model ready!')

In [ ]:
def answer_questions(label, context, questions):
    print('Context:', label)
    print('{:<48} {:<30} {:>10}'.format('Question', 'Answer', 'Confidence'))
    print('-' * 90)
    for q in questions:
        res = qa(question=q, context=context)
        print('{:<48} {:<30} {:>10.2%}'.format(q, res['answer'], res['score']))
    print()

answer_questions(
    'India',
    ('India, officially the Republic of India, is a country in South Asia. '
     'New Delhi is the capital of India. India is the most populous country '
     'in the world with over 1.4 billion people. It is known for its diverse '
     'culture, rich history, and contributions to mathematics and science. '
     'The national language is Hindi, and English is widely spoken.'),
    [
        'What is the capital of India?',
        'How many people live in India?',
        'What is the national language of India?',
    ]
)

answer_questions(
    'Python Programming',
    ('The Python programming language was created by Guido van Rossum '
     'and first released in 1991. Python emphasises code readability '
     'and supports multiple programming paradigms. It is widely used '
     'in data science, web development, and artificial intelligence.'),
    [
        'Who created Python?',
        'When was Python released?',
        'What is Python used for?',
    ]
)

---
# Program 3 — Text Summarization
**Model:** `facebook/bart-large-cnn`  
**Fix:** Uses `text2text-generation` task (compatible with all transformers versions).

In [ ]:
print('Loading summarization model...')
from transformers import BartForConditionalGeneration, BartTokenizer

model_name = 'facebook/bart-large-cnn'
tokenizer  = BartTokenizer.from_pretrained(model_name)
model      = BartForConditionalGeneration.from_pretrained(model_name)

if torch.cuda.is_available():
    model = model.to('cuda')

print('Summarizer ready!')

In [ ]:
def summarize(title, text, max_len=120, min_len=30):
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    inputs = tokenizer(text.strip(), return_tensors='pt',
                       max_length=1024, truncation=True).to(dev)
    ids     = model.generate(inputs['input_ids'],
                              max_length=max_len,
                              min_length=min_len,
                              num_beams=4,
                              length_penalty=2.0,
                              early_stopping=True)
    summary = tokenizer.decode(ids[0], skip_special_tokens=True)
    orig_w  = len(text.split())
    summ_w  = len(summary.split())
    ratio   = (1 - summ_w / orig_w) * 100
    print('Topic      :', title)
    print('Summary    :', summary)
    print('Compression: {} -> {} words  ({:.0f}% reduction)'.format(orig_w, summ_w, ratio))
    print()

texts = {
    'Artificial Intelligence': (
        'Artificial Intelligence (AI) is a branch of computer science that focuses '
        'on building systems capable of performing tasks that normally require human '
        'intelligence. These tasks include visual perception, speech recognition, '
        'decision-making, and language translation. Machine learning, a subset of AI, '
        'enables systems to learn from data without being explicitly programmed. '
        'Deep learning uses neural networks with many layers to achieve breakthroughs '
        'in image recognition, NLP, and autonomous driving. AI is now embedded in '
        'everyday products such as search engines, recommendation systems, and '
        'virtual assistants like Siri and Alexa.'
    ),
    'Climate Change': (
        'Climate change refers to long-term shifts in temperatures and weather patterns. '
        'While some changes are natural, scientific evidence shows that human activities '
        'especially the burning of fossil fuels have been the main driver of climate '
        'change since the 1800s. The consequences include rising sea levels, more '
        'frequent extreme weather events, and disruptions to ecosystems worldwide. '
        'International agreements such as the Paris Accord aim to limit global warming '
        'by reducing greenhouse gas emissions globally.'
    ),
}

for title, text in texts.items():
    summarize(title, text)

---
# Program 4 — Machine Translation
**Models:** `Helsinki-NLP/opus-mt` family  
Translates English into French, Hindi, and German.

In [ ]:
print('Loading 3 translation models...')
translators = {
    'French' : pipeline('translation_en_to_fr', model='Helsinki-NLP/opus-mt-en-fr'),
    'Hindi'  : pipeline('translation',          model='Helsinki-NLP/opus-mt-en-hi'),
    'German' : pipeline('translation_en_to_de', model='Helsinki-NLP/opus-mt-en-de'),
}
print('All translation models ready!')

In [ ]:
sentences = [
    'Hello, how are you?',
    'Natural Language Processing is fascinating.',
    'I love learning new things every day.',
    'Artificial Intelligence is the future.',
]

for sentence in sentences:
    print('EN :', sentence)
    for lang, model in translators.items():
        result = model(sentence)[0]['translation_text']
        print('{} : {}'.format(lang[:2].upper(), result))
    print()

---
## All Programs Complete!

| # | Task | Model | Method |
|---|------|-------|--------|
| 1 | Chatbot | DialoGPT-small | `pipeline('text-generation')` |
| 2 | Question Answering | roberta-base-squad2 | `pipeline('question-answering')` |
| 3 | Summarization | bart-large-cnn | `BartForConditionalGeneration` (direct) |
| 4 | Translation EN→FR/HI/DE | Helsinki-NLP opus-mt | `pipeline('translation_*')` |